# Manual Run - ETL iSeller Mitra DWH (Cleaning, step by step)

> Notebook untuk menjalankan pipeline iSeller Mitra **per step**, supaya hasil tiap step bisa dicek sebelum lanjut.
>
> **Flow:**
> 1. Check duplicates di Data Lake (headers, details non-combo, key combo, type details tidak dikenal)
> 2. Load master bundling dari API iSeller Mitra (`GetProducts`). Gagal → **STOP**, tidak ada fallback
> 3. Clean Data Lake (combo-aware, hanya tabel yang terdeteksi dobel; DELETE + re-dump dalam 1 transaksi)
> 4. Re-dump DWH summary tables (selektif)
> 5. Kirim email report
>
> Detail logic: `FLOW_dedup_details_comboset.md` (folder yang sama).
>
> **Hati-hati:** notebook ini terhubung ke database **production**. Default `DRY_RUN = True`
> (tanpa email, tanpa DELETE/re-dump). Ubah ke `False` hanya kalau hasil dry run sudah dicek.
>
> Created by data-team

---
## CONFIGURATION

In [ ]:
import datetime
import sys
import os
import time

# ===================== SET MODE =============================
#    True  = Otomatis (kemarin / current_date - 1)
#    False = Manual (isi MANUAL_START_DATE & MANUAL_END_DATE sendiri)
# ============================================================

AUTO_MODE = False

# ============================================================
#  MODE MANUAL: Isi tanggal di bawah (isi hanya jika AUTO_MODE = False)
#  Cleaning ulang periode lama: jalankan per bulan (tabel DL tidak punya index)
# ============================================================

MANUAL_START_DATE = '2026-10-06'
MANUAL_END_DATE   = '2026-10-06'

# ===================== DRY RUN ==============================
#    True  = AMAN: STEP 1-3 jalan tanpa email, tanpa DELETE/re-dump DL.
#            STEP 4 (re-dump DWH) & STEP 5 (email) di-skip.
#    False = Run asli: email terkirim, DL & DWH benar-benar ditulis ulang.
# ============================================================

DRY_RUN = True

# ===================== DWH ONLY =============================
#    True  = Skip STEP 1-3 (DL sudah bersih / di-clean sebelumnya),
#            langsung re-dump 5 tabel DWH + email report.
#            (setara run_mode "dwh_only" di DAG)
#    False = Full flow (cek DL → clean DL → re-dump DWH)
#    Mau berhenti setelah STEP 3 (setara "datalake_only")? Cukup jangan jalankan cell STEP 4 & 5.
# ============================================================

DWH_ONLY = False

# ===================== API TOKEN (opsional) =================
#    Token dibaca dari: Airflow Variable -> env var access_token_mitra -> file .env.
#    Kalau notebook jalan di luar Airflow dan token belum ada, isi di sini (jangan di-commit).
# ============================================================

ACCESS_TOKEN_MITRA = ''

# ============================================================

if AUTO_MODE:
    START_DATE = END_DATE = (datetime.date.today() - datetime.timedelta(days=1)).strftime("%Y-%m-%d")
else:
    START_DATE = MANUAL_START_DATE
    END_DATE   = MANUAL_END_DATE

if ACCESS_TOKEN_MITRA:
    os.environ['access_token_mitra'] = ACCESS_TOKEN_MITRA

print(f"Mode      : {'AUTO (kemarin)' if AUTO_MODE else 'MANUAL'}")
print(f"Period    : {START_DATE} s/d {END_DATE}")
print(f"Dry run   : {DRY_RUN}" + ("   <- AMAN: tidak ada yang ditulis" if DRY_RUN else "   <- RUN ASLI: DL & DWH akan ditulis ulang"))
print(f"DWH only  : {DWH_ONLY}")

---
## LOAD CONFIG & IMPORT PIPELINE

In [ ]:
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dw = module_from_file("config_dw", "../../config/config_db_datawarehouse.py")

HERE = os.path.abspath('.')
sys.path.insert(0, os.path.join(HERE, 'pipelines'))
sys.path.insert(0, os.path.abspath(os.path.join(HERE, '..', '..', 'plugins')))

from check_duplicates import check_duplicates
from master_bundlings import load_bundling_slot_count
from pipeline_clean_duplicates import run_clean_duplicates
import redump_etl_functions as ref

print("Config & pipeline loaded successfully.")

---
## STEP 1 - CHECK DUPLICATES (Data Lake)

Pemicu lanjut ke STEP 2-3: headers dobel **ATAU** details non-combo dobel **ATAU** key combo dobel.
Key combo dobel belum tentu duplikat (komponen identik bisa sah), jadi diputuskan oleh master di STEP 3.

In [ ]:
dup_result = None
RUN_DL = False

if DWH_ONLY:
    print("DWH_ONLY = True -> STEP 1-3 dilewati.")
else:
    dup_result = check_duplicates(START_DATE, END_DATE, config_dw, send_email=not DRY_RUN)
    RUN_DL = dup_result["has_duplicates"]

    if dup_result["unknown_detail_types"]:
        print(f"\n!!! type details tidak dikenal: {dup_result['unknown_detail_types']} -> STEP 2 akan STOP.")
    print(f"\nRingkasan: headers={dup_result['total_duplicated_headers']:,} | "
          f"details non-combo={dup_result['total_duplicated_details']:,} | "
          f"details combo={dup_result['total_duplicated_details_combo']:,} | combo_only={dup_result['combo_only']}")
    print("Lanjut ke STEP 2." if RUN_DL else "Tidak ada duplikat: STEP 2-4 tidak perlu dijalankan.")

---
## STEP 2 - LOAD MASTER BUNDLING (API iSeller Mitra)

Jumlah slot per bundling dari `GetProducts`. **Gagal (token tidak ada / expired / API error) → STOP**,
Data Lake & DWH tidak disentuh. Tidak ada fallback.

In [ ]:
bundling_slot_count, master_status = None, None

if not RUN_DL:
    print("STEP 2 dilewati (tidak ada duplikat atau DWH_ONLY).")
else:
    # type details tidak dikenal -> STOP (komponen combo sah bisa ikut terhapus kalau dianggap non-combo)
    if dup_result["unknown_detail_types"]:
        dag_id, run_id, exec_at = ref._mitra_teams_meta(None)
        ref._stop_master_failed(
            START_DATE, END_DATE,
            {'ok': False, 'reason': 'unknown_detail_type',
             'message': f"type details tidak dikenal di Data Lake: {dup_result['unknown_detail_types']}", 'source': None},
            dup_result, DRY_RUN, dag_id, run_id, exec_at,
        )

    bundling_slot_count, master_status = load_bundling_slot_count()
    print(f"Source : {master_status.get('source') or '-'}")
    print(f"Status : {master_status['message']}")

    if not master_status['ok']:
        dag_id, run_id, exec_at = ref._mitra_teams_meta(None)
        ref._stop_master_failed(START_DATE, END_DATE, master_status, dup_result, DRY_RUN, dag_id, run_id, exec_at)

---
## STEP 3 - CLEAN DATA LAKE (combo-aware)

- Non-combo: 1 row per key. Combo: `rows_per_key / dup_factor`, `dup_factor` hanya dari master.
- Hanya tabel yang terdeteksi dobel di STEP 1 **dan** berubah yang ditulis ulang (1 transaksi, gagal → rollback).
- `DRY_RUN = True`: hanya hitung & tampilkan, tidak ada DELETE/re-dump.
- Baca log [3.1]–[3.5] dan `needs_manual_check` (seharusnya kosong untuk H-1).

In [ ]:
clean_stats = None

if not RUN_DL:
    print("STEP 3 dilewati (tidak ada duplikat atau DWH_ONLY).")
else:
    has_dup_headers = dup_result["total_duplicated_headers"] > 0
    has_dup_details = dup_result["total_duplicated_details"] > 0 or dup_result["total_duplicated_details_combo"] > 0

    clean_stats = run_clean_duplicates(
        START_DATE, END_DATE, config_dw,
        clean_headers=has_dup_headers,
        clean_details=has_dup_details,
        bundling_slot_count=bundling_slot_count,
        master_status=master_status,
        dry_run=DRY_RUN,
    )

---
## STEP 4 - RE-DUMP DWH SUMMARY TABLES

- Details dobel saja → 2 tabel (items, bundlings). Headers dobel → 5 tabel. `DWH_ONLY` → 5 tabel.
- Hanya key combo dobel dan master memutuskan tidak ada row dibuang → DWH tidak perlu di-re-dump.
- `DRY_RUN = True` → di-skip.

In [ ]:
results = {}

if DRY_RUN:
    print("DRY_RUN = True -> STEP 4 (re-dump DWH) di-skip.")
elif not DWH_ONLY and not RUN_DL:
    print("Tidak ada duplikat -> re-dump DWH tidak diperlukan.")
elif (not DWH_ONLY) and dup_result["combo_only"] and not (clean_stats["headers_rewritten"] or clean_stats["details_rewritten"]):
    print("Hanya key combo dobel dan master memutuskan semuanya sah (tidak ada row dibuang) -> DWH tidak perlu di-re-dump.")
else:
    has_dup_headers = True if DWH_ONLY else dup_result["total_duplicated_headers"] > 0
    pipelines = [
        ("daily_items_summary_iseller_pusat", ref.run_daily_items),
        ("daily_bundling_summary_iseller_pusat", ref.run_daily_bundlings),
    ]
    if has_dup_headers:
        pipelines += [
            ("daily_transaction_summary_iseller_pusat", ref.run_daily_transactions),
            ("hourly_transaction_summary_iseller_pusat", ref.run_hourly_transactions),
            ("daily_sales_type_summary_iseller_pusat", ref.run_daily_sales_type),
        ]

    print(f"Re-dump DWH: {len(pipelines)} of 5 pipelines | {START_DATE} s/d {END_DATE}")
    for table_name, func in pipelines:
        print()
        try:
            t0 = time.time()
            deleted, dumped, before, after = func(START_DATE, END_DATE, config_dw)
            results[table_name] = {"status": "SUCCESS", "deleted": deleted, "dumped": dumped,
                                   "time": f"{time.time() - t0:.1f}s", "before": before, "after": after}
        except Exception as e:
            results[table_name] = {"status": "FAILED", "error": str(e)}
            print(f"        ERROR: {e}")

---
## STEP 5 - SEND EMAIL REPORT

Butuh env var `GMAIL_SMTP_PASSWORD`. Hanya dikirim kalau STEP 4 menghasilkan hasil (bukan dry run).

In [ ]:
if DRY_RUN or not results:
    print("Tidak ada email report (dry run, atau STEP 4 tidak menjalankan pipeline).")
else:
    ref._send_redump_report_email(START_DATE, END_DATE, results, clean_stats)

---
## REVIEW RESULTS

In [ ]:
if clean_stats:
    s = clean_stats.get("dedup_summary") or {}
    print(f"Data Lake: headers_rewritten={clean_stats['headers_rewritten']} | details_rewritten={clean_stats['details_rewritten']}"
          f" | dry_run={clean_stats['dry_run']}")
    if s:
        print(f"Details RAW -> CLEAN: {s['details_raw']:,} -> {s['details_clean']:,} | needs_manual_check: {s['needs_manual_check_orders']:,} order")
    print()

if results:
    print(f"{'Table':<55} {'Status':<10} {'Deleted':>10} {'Dumped':>10}")
    print("-" * 90)
    for table_name, r in results.items():
        if r['status'] == 'SUCCESS':
            print(f"{table_name:<55} {r['status']:<10} {r['deleted']:>10,} {r['dumped']:>10,}")
        else:
            print(f"{table_name:<55} {r['status']:<10} Error: {r['error']}")
    failed = [n for n, r in results.items() if r['status'] != 'SUCCESS']
    if failed:
        print(f"\n{len(failed)} pipeline FAILED: {', '.join(failed)}")
else:
    print("No DWH results (dry run / tidak ada yang di-re-dump).")

---
## OPSIONAL - Jalankan sekaligus (sama persis dengan DAG)

Tidak perlu kalau sudah menjalankan STEP 1–5 di atas. Di sini `skip_dl_check` = `DWH_ONLY`,
`dry_run` = `DRY_RUN`. Hati-hati: kalau `DRY_RUN = False`, ini mengirim email/Teams dan menulis ke DB.

In [ ]:
# results_all = ref.run_all(START_DATE, END_DATE, config_dw, skip_dl_check=DWH_ONLY, dry_run=DRY_RUN)